In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

df = pd.read_csv("../data/raw/customer_data.csv")
print("Before:", df.shape)

df = df.drop_duplicates().reset_index(drop=True)
print("After removing duplicates:", df.shape)

Before: (60360, 12)
After removing duplicates: (60000, 12)


In [2]:
gender_map = {"male": "Male", "m": "Male", "female": "Female", "f": "Female", "other": "Other"}
df["Gender"] = df["Gender"].str.strip().str.lower().map(gender_map)
df["City"] = df["City"].str.strip().str.title()

print(df["Gender"].value_counts(dropna=False))
print(df["City"].nunique(), "cities")

Gender
Male      29192
Female    29081
Other      1727
Name: count, dtype: int64
12 cities


In [3]:
bad_age = ~df["Age"].between(18, 100) & df["Age"].notna()
print("Impossible ages found:", bad_age.sum())
df.loc[bad_age, "Age"] = np.nan

print("Age missing now:", df["Age"].isna().sum())

Impossible ages found: 15
Age missing now: 949


In [4]:
# Number_of_Orders has min 1 in the data, so no division by zero.
df["Avg_Order_Value"] = df["Total_Spend"] / df["Number_of_Orders"]
df["Items_Per_Order"] = df["Items_Purchased"] / df["Number_of_Orders"]

df[["Total_Spend", "Number_of_Orders", "Avg_Order_Value", "Items_Per_Order"]].describe().round(1)

,Total_Spend,Number_of_Orders,Avg_Order_Value,Items_Per_Order
count,60000.0,60000.0,60000.0,60000.0
mean,35031.3,13.1,1952.2,2.2
std,52391.0,8.9,1621.3,0.7
min,186.8,1.0,182.9,1.0
25%,5594.7,6.0,930.6,1.7
50%,16636.4,12.0,1463.4,2.2
75%,36307.9,17.0,2398.4,2.6
max,1625517.4,53.0,70674.7,6.0


In [5]:
check_cols = ["Total_Spend", "Number_of_Orders", "Avg_Order_Value",
              "Items_Per_Order", "Days_Since_Last_Purchase", "Average_Rating"]
log_df = df[check_cols].copy()
for col in ["Total_Spend", "Number_of_Orders", "Avg_Order_Value", "Days_Since_Last_Purchase"]:
    log_df[col] = np.log1p(log_df[col])

log_df.corr().round(2)

,Total_Spend,Number_of_Orders,Avg_Order_Value,Items_Per_Order,Days_Since_Last_Purchase,Average_Rating
Total_Spend,1.00,0.93,0.88,0.66,-0.36,0.26
Number_of_Orders,0.93,1.00,0.64,0.56,-0.37,0.22
Avg_Order_Value,0.88,0.64,1.00,0.65,-0.29,0.27
Items_Per_Order,0.66,0.56,0.65,1.00,-0.39,0.19
Days_Since_Last_Purchase,-0.36,-0.37,-0.29,-0.39,1.00,-0.36
Average_Rating,0.26,0.22,0.27,0.19,-0.36,1.00


In [6]:
df["Discount_Applied"] = (df["Discount_Applied"].str.strip().str.lower()
                          .map({"yes": 1, "no": 0}))
print("Unmapped discount values:", df["Discount_Applied"].isna().sum())

skew_cols = ["Number_of_Orders", "Avg_Order_Value",
             "Days_Since_Last_Purchase", "Items_Per_Order"]
skew_table = pd.DataFrame({
    "skew_before": df[skew_cols].skew(),
    "skew_after_log1p": np.log1p(df[skew_cols]).skew(),
})
skew_table.round(2)

Unmapped discount values: 0


,skew_before,skew_after_log1p
Number_of_Orders,0.93,-0.48
Avg_Order_Value,4.92,0.24
Days_Since_Last_Purchase,1.67,-0.21
Items_Per_Order,0.08,-0.38
